In [1]:
import pandas as pd
import numpy as np
import networkx as nx

In [3]:
# Subsitute path with string raw folder
# Here I'm using string v12.0  

data_path="/home/ubuntu/Data/"

string_df = pd.read_csv(data_path+"networks/9606.protein.links.detailed.v12.0.txt", sep=" ")
string_df.head(1)

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
0,9606.ENSP00000000233,9606.ENSP00000356607,0,0,0,45,134,0,81,173


In [5]:
string_alias = pd.read_csv(data_path+"networks/9606.protein.aliases.v12.0.txt", sep="\t")
string_alias.head(1)

,#string_protein_id,alias,source
0,9606.ENSP00000000233,2B6H,Ensembl_PDB


In [6]:
#Ensembl_HGNC_ensembl_gene_id
#Ensembl
#Ensembl_gene

string_alias[string_alias['source'] == 'Ensembl_gene'].shape

(19699, 3)

In [7]:
# Filter for data_quality 
string_df[(string_df['database'] > 400) | (string_df['experimental'] > 400)]

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
16,9606.ENSP00000000233,9606.ENSP00000354878,0,0,0,42,0,500,66,513
26,9606.ENSP00000000233,9606.ENSP00000310226,0,0,0,109,110,500,218,648
34,9606.ENSP00000000233,9606.ENSP00000363232,0,0,0,0,105,500,0,533
61,9606.ENSP00000000233,9606.ENSP00000262455,0,0,0,94,543,0,73,582
80,9606.ENSP00000000233,9606.ENSP00000405926,0,0,0,98,114,500,0,565
...,...,...,...,...,...,...,...,...,...,...
13715148,9606.ENSP00000501317,9606.ENSP00000372023,0,0,0,57,441,0,116,493
13715149,9606.ENSP00000501317,9606.ENSP00000340373,0,0,0,67,510,0,90,547
13715245,9606.ENSP00000501317,9606.ENSP00000343741,0,0,0,49,514,0,116,555
13715299,9606.ENSP00000501317,9606.ENSP00000305071,0,0,0,0,965,0,386,978


In [8]:
filtered_string = string_df[string_df['combined_score'] > 700]
filtered_string.shape

(472000, 10)

In [9]:
def compute_prior_away(score, prior):

    if score < prior: score = prior
    score_no_prior = (score - prior) / (1 - prior)

    return score_no_prior


def calculate_combined(x):

    prior = 0.041

    ## divide by 1000

    neighborhood = float(x['neighborhood']) / 1000
    fusion = float(x['fusion']) / 1000
    cooccurrence =  float(x['cooccurence']) / 1000
    coexpression = float(x['coexpression']) / 1000
    experiments = float(x['experimental']) / 1000
    database = float(x['database']) / 1000


    ## compute prior away

    neighborhood_prior_corrected                 = compute_prior_away (neighborhood, prior)             
    fusion_prior_corrected                       = compute_prior_away (fusion, prior)             
    cooccurrence_prior_corrected                 = compute_prior_away (cooccurrence, prior)           
    coexpression_prior_corrected                 = compute_prior_away (coexpression, prior)            
    experiments_prior_corrected                  = compute_prior_away (experiments, prior)   
    database_prior_corrected                     = compute_prior_away (database, prior)       

    ## then, combine the direct and transferred scores for each category:

    neighborhood_both_prior_corrected = 1.0 - (1.0 - neighborhood_prior_corrected) 
    coexpression_both_prior_corrected = 1.0 - (1.0 - coexpression_prior_corrected) 
    experiments_both_prior_corrected = 1.0 - (1.0 - experiments_prior_corrected) 
    database_both_prior_corrected = 1.0 - (1.0 - database_prior_corrected)

    ## next, do the 1 - multiplication:
    combined_score_one_minus = (
        (1.0 - experiments_both_prior_corrected) *
        (1.0 - database_both_prior_corrected)) 

    ## and lastly, do the 1 - conversion again, and put back the prior *exactly once*

    combined_score = (1.0 - combined_score_one_minus)            ## 1- conversion
    combined_score *= (1.0 - prior)                              ## scale down
    combined_score += prior                                      ## and add prior.

    ## round

    combined_score = int(combined_score * 1000)
    return combined_score

In [10]:
string_df['new_combined'] = string_df.apply(lambda x: calculate_combined(x), axis = 1)

In [11]:
filtered_string = string_df[string_df['new_combined'] >= 700]
print(len(filtered_string))

225606


In [12]:
string_alias[string_alias['source'] == 'Ensembl_HGNC_entrez_id']

,#string_protein_id,alias,source
2,9606.ENSP00000000233,381,Ensembl_HGNC_entrez_id
118,9606.ENSP00000000412,4074,Ensembl_HGNC_entrez_id
318,9606.ENSP00000001008,2288,Ensembl_HGNC_entrez_id
493,9606.ENSP00000001146,56603,Ensembl_HGNC_entrez_id
652,9606.ENSP00000002125,55471,Ensembl_HGNC_entrez_id
...,...,...,...
3888480,9606.ENSP00000501254,222,Ensembl_HGNC_entrez_id
3888621,9606.ENSP00000501259,642968,Ensembl_HGNC_entrez_id
3888730,9606.ENSP00000501265,284434,Ensembl_HGNC_entrez_id
3888903,9606.ENSP00000501277,8861,Ensembl_HGNC_entrez_id


In [13]:
filtered_string_ensembl_df = (
    filtered_string
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_gene'].add_suffix('_prot1'), 
        left_on="protein1",
        right_on="#string_protein_id_prot1",
        validate='many_to_one'
    )
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_gene'].add_suffix('_prot2'), 
        left_on="protein2",
        right_on="#string_protein_id_prot2",
        validate='many_to_one'
    )
    .merge(
        string_alias[string_alias['source'] == 'UniProt_DR_GeneID'].drop_duplicates('#string_protein_id').add_suffix('_ent1'), 
        left_on="protein1",
        right_on="#string_protein_id_ent1",
        validate='many_to_one',
        how='left'
    )
    .merge(
        string_alias[string_alias['source'] == 'UniProt_DR_GeneID'].drop_duplicates('#string_protein_id').add_suffix('_ent2'), 
        left_on="protein2",
        right_on="#string_protein_id_ent2",
        validate='many_to_one',
        how='left'
    )
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_HGNC'].drop_duplicates('#string_protein_id').add_suffix('_sym1'), 
        left_on="protein1",
        right_on="#string_protein_id_sym1",
        validate='many_to_one',
        how='left'
    )
    .merge(
        string_alias[string_alias['source'] == 'Ensembl_HGNC'].drop_duplicates('#string_protein_id').add_suffix('_sym2'), 
        left_on="protein2",
        right_on="#string_protein_id_sym2",
        validate='many_to_one',
        how='left'
    )
    .drop(
        ['#string_protein_id_prot1','#string_protein_id_prot2','source_prot1','source_prot2',
         '#string_protein_id_ent2','#string_protein_id_ent1','#string_protein_id_sym2','#string_protein_id_sym1'],
        axis='columns'  
    )
)

# number of rows does not change 
assert filtered_string_ensembl_df.shape[0] == filtered_string.shape[0] # FAILS as some have no symbol 
filtered_string_ensembl_df.head(2)

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score,...,alias_prot1,alias_prot2,alias_ent1,source_ent1,alias_ent2,source_ent2,alias_sym1,source_sym1,alias_sym2,source_sym2
0,9606.ENSP00000000233,9606.ENSP00000256682,0,0,0,128,477,500,60,756,...,ENSG00000004059,ENSG00000134287,381,UniProt_DR_GeneID,377,UniProt_DR_GeneID,ARF5,Ensembl_HGNC,ARF3,Ensembl_HGNC
1,9606.ENSP00000000233,9606.ENSP00000306010,0,0,0,49,780,500,57,888,...,ENSG00000004059,ENSG00000168374,381,UniProt_DR_GeneID,378,UniProt_DR_GeneID,ARF5,Ensembl_HGNC,ARF4,Ensembl_HGNC


In [14]:
columns_selection = [
    'protein1',
    'protein2',
    'combined_score',
    'alias_prot1',
    'alias_prot2',
    'alias_ent1',
    'alias_ent2',
    'alias_sym1',
    'alias_sym2'
]

filtered_string_ensembl_df[columns_selection].to_parquet("../data/quality_control/string_filtered.parquet",compression="zstd")